In [0]:
df = spark.table("bankingdata.bronze.account")

In [0]:
df.columns

### Handling rescue data column

In [0]:
from pyspark.sql.functions import *

In [0]:
df_rescued = df.filter(col("_rescued_data").isNotNull())
df = df.filter(col("_rescued_data").isNull())

In [0]:
df_rescued_final = df_rescued.select(
    col("AccountID"),          
    col("_rescued_data"),       
    col("source_file"),        
    col("ingestion_time")       
)

In [0]:
df_rescued_final.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bankingdata.rescueddata.account_rescued")

### Required Columns

In [0]:
df = df.select(
    col("AccountID").cast("int"),
    col("CustomerID").cast("int"),
    col("BranchID").cast("int"),
    col("AccountType").cast("string"),
    col("Balance").cast("double"),
    col("Status").cast("string"),
    col("ModifiedDate").cast("timestamp")
)

### Schema

In [0]:
df = df.select(
    col("AccountID").cast("int").alias("AccountID"),
    col("CustomerID").cast("int").alias("CustomerID"),
    col("BranchID").cast("int").alias("BranchID"),
    col("AccountType").cast("string").alias("AccountType"),
    col("Balance").cast("double").alias("Balance"),
    col("Status").cast("string").alias("Status"),
    col("ModifiedDate").cast("timestamp").alias("ModifiedDate")
)

### Primary Key Validation

In [0]:
df_valid = df.filter(
    col("AccountID").isNotNull() &                    
    (trim(col("AccountID").cast("string")) != "") &  
    (col("AccountID") > 0)                          
)

### Business Quality Check


In [0]:
df = df.filter(

    # CustomerID (FK)
    col("CustomerID").isNotNull() &
    (col("CustomerID") > 0) &

    # BranchID (FK)
    col("BranchID").isNotNull() &
    (col("BranchID") > 0) &

    # AccountType
    col("AccountType").isNotNull() &
    col("AccountType").isin("Savings", "Current", "Credit") &

    # Balance
    col("Balance").isNotNull() &
    (col("Balance") >= 0) &

    # Status
    col("Status").isNotNull() &
    col("Status").isin("Active", "Inactive", "Closed") &

    # ModifiedDate
    col("ModifiedDate").isNotNull() &
    (col("ModifiedDate") <= current_timestamp())
)

### Remove Duplicates

In [0]:
from pyspark.sql.window import Window

# Window definition
window_spec = Window.partitionBy("AccountID") \
                    .orderBy(col("ModifiedDate").desc())

# Dedup logic
df_dedup = df_valid.withColumn("rn", row_number().over(window_spec)) \
                   .filter(col("rn") == 1) \
                   .drop("rn")

### Delta table

In [0]:
# save table
silver_table = "bankingdata.silver.account"

df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(silver_table)

print("Silver table created:", silver_table)